#  Mail RAG – RAG chat over a mail knowledge base

You can access the mails knowledge base here - https://drive.google.com/drive/folders/1rmWBy2Mon-8_l0RHNFNqRK8yG-Y1qR2X?usp=sharing

Trying out RAG with LANGCHAIN. This project:

1. Loads markdown “mail” docs from knowledge-base/, chunks them with LangChain’s RecursiveCharacterTextSplitter, and embeds with HuggingFace all-MiniLM-L6-v2 into Chroma.

2. Adds 2D/3D t-SNE (Plotly) visualizations of the vector store by doc type (main, promotion, purchases, others).

3. Implements RAG: retriever over Chroma + gpt-4.1-nano (OpenRouter); combines prior user messages with the current question for retrieval and passes full chat history to the LLM for follow-ups.

4. Uses a Gradio ChatInterface for Q&A over the mail corpus with conversation history.



### PART A: Divide our documents into chunks

In [24]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import gradio as gr

In [25]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_url = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenRouter API Key not set")


OpenRouter API Key exists and begins sk-or-v1


In [26]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 20 files in the knowledge base
Total characters in knowledge base: 6,645


In [27]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 1,722


In [28]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 20 documents


In [29]:
documents[1]

Document(metadata={'source': 'knowledge-base/promotion/loyalty-reward-reminder.md', 'doc_type': 'promotion'}, page_content='# Your loyalty reward is about to expire\n\n**From:** rewards@loyalty.com  \n**To:** you@email.com  \n**Date:** 2024-03-12  \n**Category:** promotion\n\nYou have 500 points expiring on March 31. Redeem them for a $10 discount, free shipping, or donate to our partner charity. Log in to your account to choose an option.\n\nLoyalty Team\n')

In [30]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 20 chunks
First chunk:

page_content='# March newsletter – new arrivals and tips

**From:** newsletter@brand.com  
**To:** you@email.com  
**Date:** 2024-03-01  
**Category:** promotion

This month: new arrivals in our Home collection, 5 styling tips from our editors, and an exclusive subscriber offer – 15% off your next order with code MARCH15. Expires March 31.

Unsubscribe or update preferences in your account.' metadata={'source': 'knowledge-base/promotion/newsletter-march.md', 'doc_type': 'promotion'}


In [31]:
chunks[18]

Document(metadata={'source': 'knowledge-base/main/office-reopening.md', 'doc_type': 'main'}, page_content='# Office reopening – hybrid policy\n\n**From:** hr@company.com  \n**To:** all@company.com  \n**Date:** 2024-02-20  \n**Category:** main\n\nThe office will reopen for hybrid work from March 1. You can work from the office up to 3 days per week if you prefer. Desk booking is via the intranet. Core hours for in-person meetings remain 10am–3pm.\n\nHR')

### PART B: Make vectors and store in Chroma


In [ ]:

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 20 documents


In [ ]:
# Let's see the number of vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 20 vectors with 384 dimensions in the vector store


### Part C: Visualize!

In [34]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['main', 'promotion', 'purchases', 'others'].index(t)] for t in doc_types]

In [35]:
# visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)
# Perplexity must be < n_samples; for small datasets use min(30, n_samples - 1)
n_samples = len(vectors)
tsne = TSNE(n_components=2, random_state=18, perplexity=min(30, n_samples - 1))
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [36]:
# Let's try 3D!
# Perplexity must be < n_samples for t-SNE
tsne = TSNE(n_components=3, random_state=42, perplexity=min(30, len(vectors) - 1))
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

### Set up the 2 key LangChain objects: retriever and llm


In [38]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(base_url=openrouter_url, api_key=openrouter_api_key, temperature=0, model_name=MODEL)

In [39]:
retriever.invoke("What is my last mail?")

[Document(id='5f33eec3-2e90-436b-a130-b911a47c33aa', metadata={'source': 'knowledge-base/purchases/shipping-update-1002.md', 'doc_type': 'purchases'}, page_content='# Your order #1002 has shipped\n\n**From:** shipping@store.com  \n**To:** you@email.com  \n**Date:** 2024-03-11  \n**Category:** purchases\n\nOrder **#1002** is on its way. Carrier: FastShip. Tracking number: FS7890123456. Expected delivery: March 13. You can track the package on the carrier’s website or in your account.\n\nStore Team'),
 Document(id='2f550570-b391-48b0-9b5d-a3df18a16326', metadata={'source': 'knowledge-base/purchases/order-delivered-1000.md', 'doc_type': 'purchases'}, page_content='# Order #1000 delivered\n\n**From:** delivery@store.com  \n**To:** you@email.com  \n**Date:** 2024-03-05  \n**Category:** purchases\n\nOrder **#1000** was delivered on March 5. We hope you’re happy with your purchase. If anything isn’t right, contact us within 30 days for a return or exchange.\n\nStore Team'),
 Document(id='2d69

In [40]:
llm.invoke("hat is my last mail?")

AIMessage(content="I don't have access to your emails or personal information. If you'd like, you can share the content of your last email here, and I can help you with it.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 13, 'total_tokens': 48, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 1.53e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.53e-05, 'upstream_inference_prompt_cost': 1.3e-06, 'upstream_inference_completions_cost': 1.4e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4.1-nano-2025-04-14', 'system_fingerprint': None, 'id': 'gen-1772519206-BALxIc0fsO4TMbDCA3Y1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--cc4c189a-2125-4608-8cb9-8

## Creating the Chat Ui!

In [41]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant helping a user with their mail.
You are chatting with a user about their mail.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [46]:
def combined_question(question: str, history: list = []) -> str:
    """
    Combine all the user's messages into a single string.
    Gradio ChatInterface passes history as list of (user_msg, assistant_msg) tuples.
    """
    def user_text(m):
        if isinstance(m, (list, tuple)) and len(m) >= 1:
            return m[0]  # Gradio tuple format: (user, assistant)
        if isinstance(m, dict):
            return m.get("content", "") if m.get("role") == "user" else ""
        return ""
    prior = "\n".join(user_text(m) for m in history if user_text(m))
    return (prior + "\n" + question) if prior else question

In [ ]:
def answer_question(question: str, history):
    combined = combined_question(question, history)
    docs = retriever.invoke(combined)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    # Build message list so the LLM sees the conversation history (for follow-up questions)
    messages = [SystemMessage(content=system_prompt)]
    for m in history:
        if isinstance(m, (list, tuple)) and len(m) >= 2:
            messages.append(HumanMessage(content=m[0]))
            messages.append(AIMessage(content=m[1]))
        elif isinstance(m, dict):
            if m.get("role") == "user":
                messages.append(HumanMessage(content=m.get("content", "")))
            elif m.get("role") == "assistant":
                messages.append(AIMessage(content=m.get("content", "")))
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    return response.content

In [44]:
answer_question("Who is Acme Corp?", [])

'Acme Corp is a client that recently provided positive feedback on a delivery and expressed interest in expanding their contract in the upcoming quarter.'

In [53]:
gr.ChatInterface(answer_question).launch(inbrowser=True)

/Users/admin/Documents/ai_engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning:

The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.



* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
